In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
from transformers import AutoModelForTokenClassification, AutoTokenizer,DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer
import torch
import evaluate  # pip install evaluate
import seqeval   # pip install seqeval
from datasets import load_dataset
import numpy as np # linear algebra


In [6]:
# 加载hf中dataset
ds = load_dataset('doushabao4766/msra_ner_k_V3')
ds

README.md:   0%|          | 0.00/697 [00:00<?, ?B/s]

data/train-00000-of-00001-42717a92413393(…):   0%|          | 0.00/13.9M [00:00<?, ?B/s]

data/test-00000-of-00001-8899cab5fdab45b(…):   0%|          | 0.00/946k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45001 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3443 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'knowledge'],
        num_rows: 45001
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'knowledge'],
        num_rows: 3443
    })
})

In [7]:
# 减小数据量
ds = ds['train'].train_test_split(
    test_size=0.3,  # 10%作为测试集
    seed=42,        # 随机种子，确保可重复
    shuffle=True,    # 是否打乱
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-chinese')

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
def tokenize_fn(examples):
    result = tokenizer(
        examples['tokens'], 
        max_length=512, 
        truncation=True, 
        add_special_tokens=False,
        is_split_into_words=True
    )
    
    labels = []
    for i, ner_tags in enumerate(examples['ner_tags']):
        word_ids = result.word_ids(batch_index=i)
        labels.append([ner_tags[w] for w in word_ids])
    
    result['labels'] = labels
    return result

ds1 = ds.map(tokenize_fn, batched=True)


Map:   0%|          | 0/31500 [00:00<?, ? examples/s]

Map:   0%|          | 0/13501 [00:00<?, ? examples/s]

In [10]:
ds1['train'].features

{'id': Value('string'),
 'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])),
 'knowledge': Value('string'),
 'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8')),
 'labels': List(Value('int64'))}

In [11]:
ds1.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'labels'])

In [12]:
ds1

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'knowledge', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 31500
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'knowledge', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 13501
    })
})

In [13]:
for item in ds1['train']:
    print(item['labels'])
    print(item['input_ids'])
    break

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
tensor([8034, 8026, 1905, 4415,  712,  818, 1999, 1447,  833, 6379,  769, 1215,
        4638, 1071,  800, 2339,  868,  511])


In [14]:
args = TrainingArguments(
    output_dir="ner_train",  # 模型训练工作目录（tensorboard，临时模型存盘文件，日志）
    num_train_epochs = 3,    # 训练 epoch
    per_device_train_batch_size=32,  # 训练批次
    per_device_eval_batch_size=32,
    report_to='tensorboard',  # 训练输出记录
    eval_strategy="epoch"
)

In [15]:
tags = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

In [16]:
tags

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

In [17]:
id2lbl = {i:tag for i, tag in enumerate(tags)}
lbl2id = {tag:i for i, tag in enumerate(tags)}

model = AutoModelForTokenClassification.from_pretrained('google-bert/bert-base-chinese', 
                                                        num_labels=7,
                                                        id2label=id2lbl,
                                                        label2id=lbl2id)
model

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: google-bert/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly in

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [18]:
ds['train'].features

{'id': Value('string'),
 'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])),
 'knowledge': Value('string')}

In [19]:
import numpy as np


def filter_and_convert(predicts, labels, tags):
    """过滤填充标签并转换为标签名称"""
    predictions_list = []
    labels_list = []
    
    for ps, ls in zip(predicts, labels):
        # 同时处理预测和标签，避免重复计算
        pred_tokens = []
        label_tokens = []
        for p, l in zip(ps, ls):
            if l != -100:  # 过滤填充标签
                pred_tokens.append(tags[p])
                label_tokens.append(tags[l])
        predictions_list.append(pred_tokens)
        labels_list.append(label_tokens)
    
    return predictions_list, labels_list

# metric 方法
def compute_metric(result):
    # result 是一个tuple (predicts, labels)

    # 获取评估对象
    seqeval = evaluate.load('seqeval')
    predicts, labels = result
    predicts = np.argmax(predicts, axis=-1)

    predictions_list, labels_list = filter_and_convert(predicts, labels, tags)
    results = seqeval.compute(predictions=predictions_list, references=labels_list)

    return results

In [20]:
ds1

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'knowledge', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 31500
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'knowledge', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 13501
    })
})

In [21]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True)

trainer = Trainer(
    model,
    args,
    train_dataset=ds1['train'],
    eval_dataset=ds1['test'],
    data_collator=data_collator,
    compute_metrics=compute_metric
)

2026-03-15 09:23:45.201645: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773566625.355860      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773566625.402412      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773566625.795194      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773566625.795233      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773566625.795236      55 computation_placer.cc:177] computation placer alr

In [22]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Loc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,No log,0.051221,"{'precision': 0.9275256222547584, 'recall': 0.9311897106109325, 'f1': 0.9293540549213772, 'number': 10885}","{'precision': 0.8429739075762339, 'recall': 0.8734527687296417, 'f1': 0.8579427291633339, 'number': 6140}","{'precision': 0.9663274745605921, 'recall': 0.9728068541627863, 'f1': 0.9695563393354373, 'number': 5369}",0.913065,0.925337,0.919160,0.991774
2,0.118119,0.045298,"{'precision': 0.9465607015620718, 'recall': 0.9519522278364723, 'f1': 0.9492488090875779, 'number': 10885}","{'precision': 0.8989833790543812, 'recall': 0.907328990228013, 'f1': 0.9031369052443868, 'number': 6140}","{'precision': 0.9805170475833646, 'recall': 0.9748556528217546, 'f1': 0.9776781544783786, 'number': 5369}",0.941509,0.945209,0.943355,0.993553
3,0.030379,0.048846,"{'precision': 0.947478229317852, 'recall': 0.9595774000918695, 'f1': 0.9534894335661145, 'number': 10885}","{'precision': 0.9000793021411578, 'recall': 0.9242671009771987, 'f1': 0.9120128565689032, 'number': 6140}","{'precision': 0.9786364480772803, 'recall': 0.9811883032222015, 'f1': 0.9799107142857143, 'number': 5369}",0.941705,0.955077,0.948344,0.994074


Trainer is attempting to log a value of "{'precision': 0.9275256222547584, 'recall': 0.9311897106109325, 'f1': 0.9293540549213772, 'number': 10885}" of type <class 'dict'> for key "eval/LOC" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'precision': 0.8429739075762339, 'recall': 0.8734527687296417, 'f1': 0.8579427291633339, 'number': 6140}" of type <class 'dict'> for key "eval/ORG" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'precision': 0.9663274745605921, 'recall': 0.9728068541627863, 'f1': 0.9695563393354373, 'number': 5369}" of type <class 'dict'> for key "eval/PER" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Trainer is attempting to log a value of "{'precision': 0.9465607015620718, 'recall': 0.9519522278364723, 'f1': 0.9492488090875779, 'number': 10885}" of type <class 'dict'> for key "eval/LOC" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'precision': 0.8989833790543812, 'recall': 0.907328990228013, 'f1': 0.9031369052443868, 'number': 6140}" of type <class 'dict'> for key "eval/ORG" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'precision': 0.9805170475833646, 'recall': 0.9748556528217546, 'f1': 0.9776781544783786, 'number': 5369}" of type <class 'dict'> for ke

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Trainer is attempting to log a value of "{'precision': 0.947478229317852, 'recall': 0.9595774000918695, 'f1': 0.9534894335661145, 'number': 10885}" of type <class 'dict'> for key "eval/LOC" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'precision': 0.9000793021411578, 'recall': 0.9242671009771987, 'f1': 0.9120128565689032, 'number': 6140}" of type <class 'dict'> for key "eval/ORG" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'precision': 0.9786364480772803, 'recall': 0.9811883032222015, 'f1': 0.9799107142857143, 'number': 5369}" of type <class 'dict'> for ke

TrainOutput(global_step=1479, training_loss=0.05438721897313529, metrics={'train_runtime': 1746.3995, 'train_samples_per_second': 54.111, 'train_steps_per_second': 0.847, 'total_flos': 8406976461662856.0, 'train_loss': 0.05438721897313529, 'epoch': 3.0})

In [23]:
input_data = '双方确定了今后发展中美关系的指导方针。'

input_tokens = tokenizer(
    input_data, 
    add_special_tokens=False
)


In [24]:
input_tokens

{'input_ids': [1352, 3175, 4802, 2137, 749, 791, 1400, 1355, 2245, 704, 5401, 1068, 5143, 4638, 2900, 2193, 3175, 7151, 511], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [26]:
result = trainer.predict([input_tokens])


In [32]:
predictions = np.argmax(result.predictions, axis=-1)


In [33]:
predictions

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0]])

In [35]:
input_ids = input_tokens['input_ids']


res = []

for input_id, predict in zip(input_ids, predictions[0]):
    if predict != 0:
        res.append({
            "entity": id2lbl[predict],
            "content": tokenizer.decode(input_id)
        })
print(res)

[{'entity': 'B-LOC', 'content': '中'}, {'entity': 'B-LOC', 'content': '美'}]


In [37]:
import collections

container = collections.Counter()

for item in ds['train']:
    
    if '中' in item['tokens'] or '美' in item['tokens']:
        tokens = item['tokens']
        tags = item['ner_tags'] 
        for token, label in zip(tokens, tags):
            if token == '中' and label != 0:
                container[f'中-{label}'] += 1
            if token == '美' and label != 0:
                container[f'美-{label}'] += 1

    # break
print(container)

Counter({'中-5': 3574, '中-3': 1628, '美-5': 1623, '中-4': 648, '美-3': 405, '中-6': 101, '美-6': 98, '美-4': 90, '中-2': 67, '美-2': 33, '中-1': 10})


中-5, 美-5最多

tags = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

5是LOC, 符合预期, 题目要求的返回ORG不对